### Begin

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from externel.resnet_models import *
import os
import torchvision
import blackbox_model
import configs
import matplotlib.pyplot as plt
import PGFM
import numpy as np


In [ ]:
2

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load CIFAR-10 training dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

# Load CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bb_model = blackbox_model.black_box_model_class()

correct = 0
total = 0

data_all = None
label_all = None
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        # predicted = bb_model.predict(inputs)
        predicted_prob = bb_model.predict_proba(inputs)
        predicted = torch.argmax(predicted_prob, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        if data_all is None:
            data_all = inputs
            label_all = labels
        else:
            data_all = torch.cat((data_all, inputs), dim=0)
            label_all = torch.cat((label_all, labels), dim=0)

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")
data_all = data_all.to(configs.device)
label_all = label_all.to(configs.device)

In [ ]:
train_rate = 0.8

adv_data_train = data_all[:int(train_rate * len(data_all))]
adv_data_test = data_all[int(train_rate * len(data_all)):]
adv_label_train = label_all[:int(train_rate * len(label_all))]
adv_label_test = label_all[int(train_rate * len(label_all)):]

cifar_classes = ['airplanes', 'cars', 'birds', 'cats', 'deer', 'dogs', 'frogs', 'horses', 'ships', 'trucks']


In [ ]:

PGFM_class = PGFM.PGFM(bb_model)


In [ ]:


def imshow(img, mean = torch.tensor([0.4914, 0.4822, 0.4465]), std = torch.tensor([0.2023, 0.1994, 0.2010])):
    img = img.squeeze(0)  # Remove batch dimension
    img = img.numpy().transpose((1, 2, 0))  # Convert to HWC format
    img = img * std.numpy() + mean.numpy()  # Denormalize
    img = np.clip(img, 0, 1)  # Clip values to be in range [0,1] for imshow
    plt.imshow(img)
    plt.axis('off')  # Hide axis
    # plt.show()

stage1_t = 1
ref = adv_data_test
x_prev = torch.randn(ref.shape[0], 1, 32, 32, dtype=torch.float32, device=configs.device)


t_tensor_N = stage1_t * torch.ones(x_prev.shape[0], device=configs.device, dtype=torch.float32)
res = PGFM_class.sample_xt_given_x1_x0(x_prev, ref, t_tensor_N)
# res = ref+torch.randn_like(ref)*0.19

l2norm = torch.norm(res - adv_data_test, p=2, dim = (1,2,3))
testceloss = torch.mean(bb_model.CEloss(res.to(device), adv_label_test.to(device)))
# testfid = compute_fid(ref.to(device), res.to(device))

with torch.no_grad():
    # for inputs, labels in data_loader:
    inputs, labels = res.to(device), adv_label_test.to(device)
    predicted = bb_model.predict(inputs)
    total = labels.size(0)
    correct = (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")
print('Mean l2norm:', l2norm.mean().item())
print('CE loss:', testceloss.item())

i=0
plt.figure(figsize=(6,6))
for j in range(6,12):
    fig1 = adv_data_test[j].cpu()
    fig2 = res[j].cpu()
    i+=1
    plt.subplot(3,4,i)
    imshow(fig1)
    plt.title(cifar_classes[labels[j].item()] + str(labels[j].item()))

    i+=1
    plt.subplot(3,4,i)
    imshow(fig2)
    plt.title(cifar_classes[predicted[j].item()] + str(predicted[j].item()))
    plt.axis('off')
plt.show()



# print('FID:', testfid)

In [ ]:
# source_data, source_label, res = (
# PGFM_class.train2_2stage(adv_data_train, adv_label_train, './saved_model/FMworef_10000.pth')
PGFM_class.train2_2stage(adv_data_train, adv_label_train, './saved_model/Apr19PGFM_0p815s_iter_train_280000.pth')#FMworef_150000
# './saved_model/FMworef_10000.pth'
# PGFM_0p730s_iter_train_30000

In [ ]:
PGFM_class.train_2stage(adv_data_train, adv_label_train, init_ckpt_path=None)#'./saved_model/FMworef_150000.pth'